# 97 — Campaign B post-M2 pipeline（95 相当の消化レーン）

**既に溜まっている M2-ready / READY_FOR_M3 を消化**する（notebook 95 と同じ consumer）。

- デフォルト: `drain_existing_backlog=True` / `skip_screening=True`
  → `advance → M3 → pre_m6 → obligations → M6`（`run_pipeline_to_m6`）
- 対象: SELECTED で `ADVANCE.json` が `READY_FOR_M3`、または `m2_binding` が `READY` / `READY_SHARED`
- `M2_READY.json` は検出のみ（待機ゲートではない）
- screening を回したいときだけ `DRAIN_EXISTING_BACKLOG=False`（end_to_end 経路）
- **96 と同時起動しない**（GPU 競合）

設計: `docs/campaign_b_parallel_split_design.md`


In [ ]:
from pathlib import Path
import os, sys, json

BUNDLE_NAME = 'validated_4d_su2_rg_codex_bundle'
explicit = os.environ.get('VALIDATED_RG_PROJECT_ROOT')
candidates = ([Path(explicit)] if explicit else []) + [
    Path.cwd(), Path.cwd().parent, Path.cwd() / BUNDLE_NAME,
    Path('/notebooks') / BUNDLE_NAME, Path('/notebooks'),
]
PROJECT_ROOT = next((
    p.expanduser().resolve()
    for p in candidates
    if (p.expanduser() / 'src' / 'campaign_b' / 'post_m2_pipeline.py').is_file()
), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Pull latest main; src/campaign_b/post_m2_pipeline.py not found.')
PERSIST_ROOT = Path(
    os.environ.get('VALIDATED_RG_PERSIST_ROOT', '/storage/validated_4d_su2_rg')
).expanduser().resolve()
os.environ['VALIDATED_RG_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['VALIDATED_RG_PERSIST_ROOT'] = str(PERSIST_ROOT)
os.environ['VALIDATED_RG_DISABLE_SESSION_WALLCLOCK'] = '1'
os.environ.setdefault('VALIDATED_RG_PERSIST_ACK', 'I_CONFIRM_THIS_PATH_IS_PERSISTENT')
os.environ.setdefault('VALIDATED_RG_M3_ALLOW_CODE_DRIFT', '1')
os.environ.setdefault('VALIDATED_RG_M4_ALLOW_CODE_DRIFT', '1')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
print('PROJECT_ROOT', PROJECT_ROOT)
print('PERSIST_ROOT', PERSIST_ROOT)
if not torch.cuda.is_available():
    raise RuntimeError('Notebook 97 requires CUDA (single GPU lane).')
print('GPU', torch.cuda.get_device_properties(0).name)


In [ ]:
from src.runtime import validate_persistent_root
from src.campaign_b.post_m2_pipeline import find_m2_ready_markers

validate_persistent_root()
print('M2_READY markers (informational):', json.dumps(
    find_m2_ready_markers(PERSIST_ROOT), indent=2, default=str,
))

# Defaults = 95-equivalent drain of existing M2-ready / READY_FOR_M3 backlog.
DRAIN_EXISTING_BACKLOG = True   # False → Phase-1 end_to_end (optional screening)
SKIP_SCREENING = True           # keep True for consumer-only; ignored when draining
MAX_ROUNDS = 20                 # re-scan while backlog remains
MAX_ADVANCE = None              # None = all discovered SELECTED
MAX_M3_SESSIONS = 16
MAX_PRE_M6_PACKAGES = 16
MAX_STAGE_SESSIONS = 6
MAX_OBLIGATION_PACKAGES = 16
MAX_M6_PACKAGES = 16
MAX_QUEUE = 2000
ONLY_CAMPAIGN = None            # or pin one M7-... campaign id
# Used only when DRAIN_EXISTING_BACKLOG=False:
SELECTED_BACKLOG_TARGET = 8
SCREENING_CHUNK_SIZE = 32

print(json.dumps({
    'DRAIN_EXISTING_BACKLOG': DRAIN_EXISTING_BACKLOG,
    'SKIP_SCREENING': SKIP_SCREENING,
    'MAX_ROUNDS': MAX_ROUNDS,
    'MAX_ADVANCE': MAX_ADVANCE,
    'MAX_M3_SESSIONS': MAX_M3_SESSIONS,
    'MAX_PRE_M6_PACKAGES': MAX_PRE_M6_PACKAGES,
    'MAX_STAGE_SESSIONS': MAX_STAGE_SESSIONS,
    'MAX_OBLIGATION_PACKAGES': MAX_OBLIGATION_PACKAGES,
    'MAX_M6_PACKAGES': MAX_M6_PACKAGES,
    'MAX_QUEUE': MAX_QUEUE,
    'ONLY_CAMPAIGN': ONLY_CAMPAIGN,
}, indent=2))


In [ ]:
from src.campaign_b.post_m2_pipeline import run_post_m2_pipeline

SESSION = run_post_m2_pipeline(
    persistent_root=PERSIST_ROOT,
    project_root=PROJECT_ROOT,
    drain_existing_backlog=DRAIN_EXISTING_BACKLOG,
    skip_screening=SKIP_SCREENING,
    selected_backlog_target=SELECTED_BACKLOG_TARGET,
    screening_chunk_size=SCREENING_CHUNK_SIZE,
    max_rounds=MAX_ROUNDS,
    max_advance=MAX_ADVANCE,
    max_m3_sessions=MAX_M3_SESSIONS,
    max_pre_m6_packages=MAX_PRE_M6_PACKAGES,
    max_stage_sessions=MAX_STAGE_SESSIONS,
    max_obligation_packages=MAX_OBLIGATION_PACKAGES,
    max_m6_packages=MAX_M6_PACKAGES,
    max_queue=MAX_QUEUE,
    only_campaign_run_id=ONLY_CAMPAIGN,
)
print(json.dumps({
    'session_id': SESSION.get('session_id'),
    'mode': SESSION.get('mode'),
    'drain_existing_backlog': SESSION.get('drain_existing_backlog'),
    'skip_screening': SESSION.get('skip_screening'),
    'm2_ready_count': SESSION.get('m2_ready_count'),
    'pipeline_to_m6': SESSION.get('pipeline_to_m6'),
    'end_to_end': SESSION.get('end_to_end'),
    'ledger': str(PERSIST_ROOT / 'campaign_b' / '_post_m2' / 'LATEST_POST_M2_SESSION.json'),
    'certification_status': SESSION.get('certification_status'),
    'claim_scope': SESSION.get('claim_scope'),
    'note': SESSION.get('note'),
}, indent=2, ensure_ascii=False, default=str))
